**BTC Headline Extraction (CoinDesk)**

In [53]:
#Imports & Settings

#Core web & parsing tools
import requests
from bs4 import BeautifulSoup

#Data handling
import pandas as pd

#Progress bars & timing
import time
from tqdm import tqdm

#Speed up scraping by running multiple requests at once
from concurrent.futures import ThreadPoolExecutor, as_completed

#For saving/resuming progress
import os

In [55]:
#Settings (maybe tweak)

#Pretend to be a normal browser (helps avoid blocks)
HEADERS = {"User-Agent": "Mozilla/5.0 (academic-research)"}

#How long to wait before giving up on a webpage
TIMEOUT = 30

#How many pages to request at the same time
#If I get timeouts / failures, reduce to 5
MAX_WORKERS = 8

#Only keep articles from this date onward
MIN_DATE = "2021-01-01"

#CoinDesk sitemap index (this lists all their sitemap files)
SITEMAP_INDEX = "https://www.coindesk.com/sitemap-index.xml"

In [57]:
#Output files (saved to my folder) 
OUT_ALL_URLS = "coindesk_article_urls_all.csv"
OUT_BTC_URLS = "coindesk_bitcoin_article_urls_2021_present.csv"
OUT_CHECKPOINT = "coindesk_checkpoint_rows.csv"
OUT_FINAL = "coindesk_bitcoin_headlines_2021_present.csv"

In [59]:
#Get list of sitemap files

#Downloads the sitemap index and extracts all <loc> links (sitemap file URLs)
def get_sitemap_urls():
    r = requests.get(SITEMAP_INDEX, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()  # stops with error if request failed
    soup = BeautifulSoup(r.text, "xml")
    return [loc.text.strip() for loc in soup.find_all("loc")]

#Run it
sitemap_urls = get_sitemap_urls()
print("Sitemap files found:", len(sitemap_urls))
print("First 10 sitemap URLs:\n", sitemap_urls[:10])

Sitemap files found: 155
First 10 sitemap URLs:
 ['https://www.coindesk.com/sitemaps/static-1.xml', 'https://www.coindesk.com/sitemaps/articles-1.xml', 'https://www.coindesk.com/sitemaps/articles-2.xml', 'https://www.coindesk.com/sitemaps/articles-3.xml', 'https://www.coindesk.com/sitemaps/articles-4.xml', 'https://www.coindesk.com/sitemaps/articles-5.xml', 'https://www.coindesk.com/sitemaps/articles-6.xml', 'https://www.coindesk.com/sitemaps/articles-7.xml', 'https://www.coindesk.com/sitemaps/articles-8.xml', 'https://www.coindesk.com/sitemaps/articles-9.xml']


In [61]:
#Extract all article URLs from "articles-*.xml" sitemap files

#Each "articles-*.xml" file contains a lot of article URLs & last modified timestamps
def extract_urls_from_sitemap(sm_url):
    try:
        r = requests.get(sm_url, headers=HEADERS, timeout=TIMEOUT)
        if r.status_code != 200:
            return []
        
        sm = BeautifulSoup(r.text, "xml")

        rows = []
        for url_tag in sm.find_all("url"):
            loc = url_tag.find("loc")
            lastmod = url_tag.find("lastmod")

            #loc contains the article URL
            if loc:
                rows.append({
                    "url": loc.text.strip(),
                    "lastmod": lastmod.text.strip() if lastmod else None
                })
        return rows
    
    except Exception:
        #If anything breaks for one sitemap, just skip it
        return []

#Keep ONLY article sitemaps (ignore static pages, tags, etc.)
article_sitemaps = [u for u in sitemap_urls if "/sitemaps/articles-" in u]
print("Article sitemap files:", len(article_sitemaps))

#Loop through all article sitemap files and collect URLs
all_rows = []
for sm_url in tqdm(article_sitemaps):
    all_rows.extend(extract_urls_from_sitemap(sm_url))
    time.sleep(0.05)  #polite delay so we don’t hammer CoinDesk

#Convert to DataFrame and save
df_urls = pd.DataFrame(all_rows)
print("Total article URLs:", len(df_urls))

df_urls.to_csv(OUT_ALL_URLS, index=False)
df_urls.head()

Article sitemap files: 131


100%|████████████████████████████████████████████████████████████████████████████████| 131/131 [00:52<00:00,  2.49it/s]


Total article URLs: 65499


,url,lastmod
0,https://www.coindesk.com/markets/2026/01/12/in...,2026-01-12T06:00:11.797Z
1,https://www.coindesk.com/markets/2026/01/12/bi...,2026-01-12T05:16:10.884Z
2,https://www.coindesk.com/markets/2026/01/12/pr...,2026-01-12T04:31:23.911Z
3,https://www.coindesk.com/markets/2026/01/12/pr...,2026-01-12T02:18:14.589Z
4,https://www.coindesk.com/policy/2026/01/11/jpm...,2026-01-11T19:00:00.000Z


In [63]:
#Filter URLs to 2021+ and "bitcoin" in URL

#Load the full URL list from disk (useful if you restart the notebook)
df_urls = pd.read_csv(OUT_ALL_URLS)

#Convert lastmod to datetime
df_urls["lastmod"] = pd.to_datetime(df_urls["lastmod"], utc=True, errors="coerce")

#Keep only articles from 2021 onwards
df_2021 = df_urls[df_urls["lastmod"] >= MIN_DATE].copy()

#Keep only URLs that contain the word "bitcoin"
df_btc = df_2021[df_2021["url"].str.lower().str.contains("bitcoin", na=False)].copy()

print("URLs from 2021+:", len(df_2021))
print("Bitcoin URLs (2021+):", len(df_btc))

#Save filtered list
df_btc.to_csv(OUT_BTC_URLS, index=False)
df_btc.head()

URLs from 2021+: 40786
Bitcoin URLs (2021+): 10413


,url,lastmod
1,https://www.coindesk.com/markets/2026/01/12/bi...,2026-01-12 05:16:10.884000+00:00
13,https://www.coindesk.com/business/2026/01/10/b...,2026-01-10 15:31:54.124000+00:00
16,https://www.coindesk.com/markets/2026/01/09/as...,2026-01-10 05:16:40.544000+00:00
18,https://www.coindesk.com/markets/2026/01/09/bi...,2026-01-09 20:01:18.132000+00:00
26,https://www.coindesk.com/markets/2026/01/09/bi...,2026-01-09 13:09:27.882000+00:00


In [65]:
#Create a resusable session & define the article parser

#Using a Session is faster than requests.get repeatedly
session = requests.Session()
session.headers.update(HEADERS)

def parse_article(url):
    """
    Visits a CoinDesk article URL and extracts:
      - headline (from og:title meta tag)
      - published timestamp (from publish_date/publish_time meta tags)
    
    Returns a dict {headline, published, link} or None if parsing fails.
    """
    try:
        r = session.get(url, timeout=TIMEOUT)
        if r.status_code != 200:
            return None

        soup = BeautifulSoup(r.text, "html.parser")

        #1 Headline is stored in <meta property="og:title" content="...">
        og = soup.find("meta", {"property": "og:title"})
        headline = og.get("content").strip() if og and og.get("content") else None

        #Helper to fetch <meta name="...">
        def meta(name):
            tag = soup.find("meta", {"name": name})
            return tag.get("content").strip() if tag and tag.get("content") else None

        #2 CoinDesk stores date/time in meta tags like:
        #publish_date = 20260112
        #publish_time = 05:16
        #Some pages might use display_ or create_ instead, so we add fallbacks.
        date_str = meta("publish_date") or meta("display_date") or meta("create_date")
        time_str = meta("publish_time") or meta("display_time") or meta("create_time")

        published = f"{date_str} {time_str}" if date_str and time_str else None

        if headline and published:
            return {"headline": headline, "published": published, "link": url}

        return None

    except Exception:
        return None

In [67]:
#Mini test (10 URLs) to confirm parser works

btc_urls = pd.read_csv(OUT_BTC_URLS)["url"].tolist()

sample10 = btc_urls[:10]
rows, failed = [], 0

for u in sample10:
    item = parse_article(u)
    if item:
        rows.append(item)
    else:
        failed += 1

df_test10 = pd.DataFrame(rows)

print("TEST10 Parsed:", len(df_test10), "Failed:", failed)
df_test10.head()

TEST10 Parsed: 10 Failed: 0


,headline,published,link
0,"Bitcoin rises 1%, Nasdaq futures and dollar in...",20260112 05:16,https://www.coindesk.com/markets/2026/01/12/bi...
1,Brazilian exchange Mercado Bitcoin outlines 6 ...,20260110 15:31,https://www.coindesk.com/business/2026/01/10/b...
2,Bitcoin at $2.9 million by 2050: VanEck explai...,20260110 05:16,https://www.coindesk.com/markets/2026/01/09/as...
3,Bitcoin price news: BTC quietly retreats to $9...,20260109 20:01,https://www.coindesk.com/markets/2026/01/09/bi...
4,Bitcoin price analysis: BTC possibly poised fo...,20260109 13:09,https://www.coindesk.com/markets/2026/01/09/bi...


In [69]:
# 50 URL test (before full scrape)

sample50 = btc_urls[:50]
rows, failed = [], 0

for u in sample50:
    item = parse_article(u)
    if item:
        rows.append(item)
    else:
        failed += 1

df_test50 = pd.DataFrame(rows)

#Convert "YYYYMMDD HH:MM" into a proper datetime (UTC)
df_test50["published"] = pd.to_datetime(
    df_test50["published"],
    format="%Y%m%d %H:%M",
    utc=True,
    errors="coerce"
)

df_test50 = df_test50.dropna(subset=["published"])

print("TEST50 Parsed:", len(df_test50), "Failed:", failed)
print(df_test50.head(10))

TEST50 Parsed: 50 Failed: 0
                                            headline  \
0  Bitcoin rises 1%, Nasdaq futures and dollar in...   
1  Brazilian exchange Mercado Bitcoin outlines 6 ...   
2  Bitcoin at $2.9 million by 2050: VanEck explai...   
3  Bitcoin price news: BTC quietly retreats to $9...   
4  Bitcoin price analysis: BTC possibly poised fo...   
5  BTC price drifts around $90,000 as thin liquid...   
6  South Korea to flip bitcoin ETF stance as part...   
7  Bitcoin ETF optimism fades as three-day outflo...   
8  Bitcoin holds near $91,000 as market awaits Tr...   
9  Bitcoin price outlook: BTC's 'boring' price ac...   

                  published                                               link  
0 2026-01-12 05:16:00+00:00  https://www.coindesk.com/markets/2026/01/12/bi...  
1 2026-01-10 15:31:00+00:00  https://www.coindesk.com/business/2026/01/10/b...  
2 2026-01-10 05:16:00+00:00  https://www.coindesk.com/markets/2026/01/09/as...  
3 2026-01-09 20:01:00+00:00  ht

In [71]:
#Full scrape & checkpointing (to be able to stop/resume)

#Load all bitcoin URLs
btc_urls = pd.read_csv(OUT_BTC_URLS)["url"].tolist()

#If we already have a checkpoint file, resume from it
rows = []
done_links = set()

if os.path.exists(OUT_CHECKPOINT):
    ck = pd.read_csv(OUT_CHECKPOINT)
    rows = ck.to_dict("records")
    done_links = set(ck["link"].tolist())
    print(f"Resuming from checkpoint: {len(rows)} rows already saved")

#Only scrape URLs we haven't already done
todo = [u for u in btc_urls if u not in done_links]
print("Remaining URLs to scrape:", len(todo))

failed = 0
save_every = 300  #save progress every 300 successful parses

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(parse_article, url): url for url in todo}

    new_success = 0
    for fut in tqdm(as_completed(futures), total=len(futures)):
        item = fut.result()
        if item:
            rows.append(item)
            new_success += 1

            #Save progress regularly
            if new_success % save_every == 0:
                pd.DataFrame(rows).to_csv(OUT_CHECKPOINT, index=False)
        else:
            failed += 1

#Save final checkpoint at the end
pd.DataFrame(rows).to_csv(OUT_CHECKPOINT, index=False)

print("Scrape complete. Success:", len(rows), "Failed:", failed)

Remaining URLs to scrape: 10413


100%|████████████████████████████████████████████████████████████████████████████| 10413/10413 [21:40<00:00,  8.01it/s]

Scrape complete. Success: 9985 Failed: 428


In [81]:
#Cleaning & Saving Final Dataset

df = pd.DataFrame(rows)

# Convert published time string to datetime
df["published"] = pd.to_datetime(
    df["published"],
    format="%Y%m%d %H:%M",
    utc=True,
    errors="coerce"
)

# Drop rows where date conversion failed
df = df.dropna(subset=["published"])

# Remove duplicates
df = df.drop_duplicates(subset=["headline", "published", "link"])

# Sort chronologically
df = df.sort_values("published").reset_index(drop=True)

# Save final dataset
df.to_csv(OUT_FINAL, index=False)

print("Final saved:", len(df))
print("Date range:", df["published"].min(), "→", df["published"].max())
df.head()

Final saved: 5172
Date range: 2021-08-26 19:26:00+00:00 → 2026-01-12 05:16:00+00:00


,headline,published,link
0,How Bitcoiners Should Watch the Fed’s Jackson ...,2021-08-26 19:26:00+00:00,https://www.coindesk.com/markets/2021/08/26/ho...
1,"Ether’s Daily Issuance Drops Below Bitcoin, In...",2021-08-27 10:08:00+00:00,https://www.coindesk.com/markets/2021/08/27/et...
2,Bitcoin Holds Short-Term Support; Upside Limit...,2021-08-27 11:22:00+00:00,https://www.coindesk.com/markets/2021/08/27/bi...
3,Bill Miller’s Flagship Fund Discloses $44.7M S...,2021-08-27 16:04:00+00:00,https://www.coindesk.com/business/2021/08/27/b...
4,Bitcoin Hashrate Could Return to Record Sooner...,2021-08-27 16:44:00+00:00,https://www.coindesk.com/markets/2021/08/27/bi...


In [83]:
#Sanity Checks

df["year"] = df["published"].dt.year
print("Headlines per year:")
print(df["year"].value_counts().sort_index())

print("\nAny duplicates left?:", df.duplicated(subset=["headline", "published", "link"]).sum())


Headlines per year:
year
2021     393
2022    1029
2023    1173
2024    1051
2025    1464
2026      62
Name: count, dtype: int64

Any duplicates left?: 0


**Binance 1-minute BTCUSDT downloader & merge & 1-hour-labels**

In [88]:
#Imports & Settings

import requests
import pandas as pd
import time
from datetime import datetime, timezone

#Binance endpoint for klines (candlesticks)
BINANCE_KLINES = "https://api.binance.com/api/v3/klines"

SYMBOL = "BTCUSDT"
INTERVAL = "1m"

#Start date for price history (match your headline range)
START_DATE = "2021-01-01"

#Output files
OUT_BTC_1M = "binance_btcusdt_1m.csv"
OUT_MERGED = "coindesk_btc_merged_with_labels.csv"

In [90]:
#Helper Functions (safe requests & pagination)

def to_millis(dt_str):
    """Convert 'YYYY-MM-DD' to Unix milliseconds (UTC)."""
    dt = datetime.strptime(dt_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)

def fetch_klines(symbol, interval, start_ms, end_ms=None, limit=1000):
    """
    Fetch up to 1000 klines from Binance starting at start_ms.
    Binance returns at most 1000 rows per request.
    """
    params = {
        "symbol": symbol,
        "interval": interval,
        "startTime": start_ms,
        "limit": limit
    }
    if end_ms is not None:
        params["endTime"] = end_ms

    r = requests.get(BINANCE_KLINES, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def klines_to_df(klines):
    """
    Convert Binance kline list to a tidy DataFrame.
    """
    cols = [
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "num_trades",
        "taker_buy_base", "taker_buy_quote", "ignore"
    ]
    df = pd.DataFrame(klines, columns=cols)

    #Convert numeric columns
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    #Convert timestamps to UTC datetime
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)

    #Keep only what we need
    return df[["open_time", "open", "high", "low", "close", "volume"]]

In [94]:
# Download 1-minute BTCUSDT (resume-safe)

start_ms = to_millis(START_DATE)

#If file exists, resume from the last timestamp saved
if pd.io.common.file_exists(OUT_BTC_1M):
    existing = pd.read_csv(OUT_BTC_1M)
    existing["open_time"] = pd.to_datetime(existing["open_time"], utc=True, errors="coerce")

    last_time = existing["open_time"].max()
    print("Resuming from:", last_time)

    #Start from the next minute
    start_ms = int((last_time.value // 10**6) + 60_000)
else:
    existing = None

all_chunks = []
total_rows = 0
requests_made = 0

while True:
    klines = fetch_klines(SYMBOL, INTERVAL, start_ms, limit=1000)
    requests_made += 1

    if not klines:
        print("No more data returned. Stopping.")
        break

    df_chunk = klines_to_df(klines)
    all_chunks.append(df_chunk)
    total_rows += len(df_chunk)

    #Update start_ms to the next minute after the last returned candle
    last_open_time = df_chunk["open_time"].iloc[-1]
    start_ms = int((last_open_time.value // 10**6) + 60_000)

    #Save to disk every ~50k rows so you don’t lose progress
    if total_rows % 50000 < len(df_chunk):
        df_out = pd.concat(all_chunks, ignore_index=True)
        if existing is not None:
            df_out = pd.concat([existing, df_out], ignore_index=True)

        df_out = df_out.drop_duplicates(subset=["open_time"]).sort_values("open_time")
        df_out.to_csv(OUT_BTC_1M, index=False)
        print(f"Checkpoint saved: {len(df_out)} rows | requests: {requests_made}")

        #Reset chunk list to save RAM
        existing = df_out
        all_chunks = []

    #polite pause to avoid rate limiting
    time.sleep(0.2)

#Final save
df_out = pd.concat(all_chunks, ignore_index=True) if all_chunks else pd.DataFrame(columns=["open_time","open","high","low","close","volume"])
if existing is not None:
    df_out = pd.concat([existing, df_out], ignore_index=True)

df_out = df_out.drop_duplicates(subset=["open_time"]).sort_values("open_time")
df_out.to_csv(OUT_BTC_1M, index=False)

print("Final BTC 1m rows saved:", len(df_out))
print(df_out.head())
print(df_out.tail())

Resuming from: 2025-07-26 09:52:00+00:00
Checkpoint saved: 2450000 rows | requests: 50
Checkpoint saved: 2500000 rows | requests: 100
Checkpoint saved: 2550000 rows | requests: 150
Checkpoint saved: 2600000 rows | requests: 200
No more data returned. Stopping.
Final BTC 1m rows saved: 2646204
                  open_time      open      high       low     close     volume
0 2021-01-01 00:00:00+00:00  28923.63  28961.66  28913.12  28961.66  27.457032
1 2021-01-01 00:01:00+00:00  28961.67  29017.50  28961.01  29009.91  58.477501
2 2021-01-01 00:02:00+00:00  29009.54  29016.71  28973.58  28989.30  42.470329
3 2021-01-01 00:03:00+00:00  28989.68  28999.85  28972.33  28982.69  30.360677
4 2021-01-01 00:04:00+00:00  28982.67  28995.93  28971.80  28975.65  24.124339
                        open_time      open      high       low     close  \
2646199 2026-01-13 09:12:00+00:00  92380.25  92400.00  92380.25  92399.99   
2646200 2026-01-13 09:13:00+00:00  92400.00  92420.00  92399.99  92419.98   
2